# 03 - CNNs, Backbones e ViT

In [ ]:
import os
import random
import subprocess
import sys
from pathlib import Path

def check_numpy_runtime():
    check = subprocess.run(
        [sys.executable, '-c', 'import numpy; import numpy.random; print(numpy.__version__)'],
        capture_output=True,
        text=True,
    )
    if check.returncode == 0:
        return check.stdout.strip()

    subprocess.run([sys.executable, '-m', 'pip', 'install', '--force-reinstall', 'numpy==1.26.4'], check=True)
    raise RuntimeError('O runtime Python foi reparado. Reinicie o ambiente de execucao e rode esta celula novamente.')

numpy_version = check_numpy_runtime()

project_candidates = [Path.cwd(), Path.cwd().parent, Path('/content/ap2-ia')]
PROJECT_ROOT = next(path for path in project_candidates if (path / 'src' / 'train.py').exists())
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import numpy as np
import torch

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

print('numpy:', numpy_version)
print('torch:', torch.__version__)
print('gpu:', torch.cuda.is_available())
print('project:', PROJECT_ROOT)

In [ ]:
EPOCHS = '10'
BATCH_SIZE = '64'

def train(*args):
    command = [sys.executable, str(PROJECT_ROOT / 'src' / 'train.py'), *args, '--batch-size', BATCH_SIZE]
    print(' '.join(command))
    subprocess.run(command, cwd=PROJECT_ROOT, check=True)

experiments = [
    ['--model', 'custom_cnn', '--epochs', EPOCHS],
    ['--model', 'resnet50', '--mode', 'feature_extraction', '--optimizer', 'adamw', '--lr', '1e-3', '--epochs', EPOCHS],
    ['--model', 'efficientnet_b0', '--mode', 'feature_extraction', '--optimizer', 'adamw', '--lr', '1e-3', '--epochs', EPOCHS],
    ['--model', 'mobilenet_v3_large', '--mode', 'feature_extraction', '--optimizer', 'adamw', '--lr', '1e-3', '--epochs', EPOCHS],
    ['--model', 'vit_b_16', '--mode', 'feature_extraction', '--optimizer', 'adamw', '--lr', '1e-3', '--epochs', EPOCHS],
]

for args in experiments:
    train(*args)

In [ ]:
best_backbone = 'resnet50'

grid = [
    ['--model', best_backbone, '--mode', 'fine_tuning', '--optimizer', optimizer, '--lr', lr, '--epochs', EPOCHS]
    for optimizer in ['sgd', 'adamw']
    for lr in ['1e-2', '1e-3', '1e-4']
]

for args in grid:
    train(*args)

In [ ]:
import pandas as pd

results = pd.read_csv('experiments/results.csv')
results.sort_values('acc_val', ascending=False).head(10)